In [1]:
import torch
import numpy as np
from PIL import Image
from transformers import ViTForImageClassification, ViTImageProcessor, TrainingArguments, Trainer
from torchvision.transforms import (Compose, Normalize, RandomHorizontalFlip, RandomResizedCrop, ToTensor, Resize, CenterCrop)
from torch.utils.data import DataLoader

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
!ls /content/drive/MyDrive/images


cans  cups  pets


In [9]:
CUSTOM_DATA_ROOT = "/content/drive/MyDrive/images"  # 'can', 'pet', 'cup' 폴더가 포함된 상위 폴더

# Hugging Face datasets의 'imagefolder' 기능을 사용하여 폴더 구조를 바로 로드
# train/val/test 폴더를 명시적으로 분리하지 않았다면, 하나의 train split으로 로드됩니다.
# 이 경우, 이후 단계에서 train_test_split으로 분리해야 합니다.
from datasets import load_dataset
# 로컬 폴더를 로드할 때는 'imagefolder'와 data_dir 인수를 사용합니다.
raw_datasets = load_dataset('imagefolder', data_dir=CUSTOM_DATA_ROOT)

Resolving data files:   0%|          | 0/290 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [10]:
# 2. 훈련/검증 데이터 분리 (필수)
# 로드된 전체 데이터셋('train' split)을 90%는 훈련, 10%는 검증으로 분리
splits = raw_datasets['train'].train_test_split(test_size=0.1, seed=42)
train_ds = splits['train']
val_ds = splits['test']

In [11]:
# 3. 클래스 정의 및 매핑 (수정된 부분)
# ImageFolder 로더는 폴더 이름을 기반으로 자동으로 클래스를 정의합니다.
labels = train_ds.features['label'].names
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}
num_labels = len(labels) # 3 (can, pet, cup)

print(f"✅ 로드된 클래스: {labels}")
print(f"✅ 총 훈련 데이터 개수: {len(train_ds)}")

✅ 로드된 클래스: ['cans', 'cups', 'pets']
✅ 총 훈련 데이터 개수: 261


In [12]:
model_name_or_path = 'google/vit-base-patch16-224-in21k'
processor = ViTImageProcessor.from_pretrained(model_name_or_path)

image_mean, image_std = processor.image_mean, processor.image_std
size = processor.size["height"]

# 전처리 파이프라인 정의 (CIFAR-10 코드와 동일)
normalize = Normalize(mean=image_mean, std=image_std)
_train_transforms = Compose(
    [RandomResizedCrop(size), RandomHorizontalFlip(), ToTensor(), normalize]
)
_val_transforms = Compose(
    [Resize(size), CenterCrop(size), ToTensor(), normalize]
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [13]:
def train_transforms(examples):
    # 'image' 컬럼은 ImageFolder의 기본 출력 컬럼입니다.
    examples['pixel_values'] = [_train_transforms(image.convert("RGB")) for image in examples['image']]
    return examples

def val_transforms(examples):
    examples['pixel_values'] = [_val_transforms(image.convert("RGB")) for image in examples['image']]
    return examples

In [14]:
# 전처리 적용
train_ds.set_transform(train_transforms)
val_ds.set_transform(val_transforms)

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

model = ViTForImageClassification.from_pretrained(
    model_name_or_path,
    num_labels=num_labels, # 3으로 자동 설정됨
    id2label=id2label,
    label2id=label2id
)

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
# =======================================================
# 📌 5. Trainer 설정 및 학습 실행 (기존 코드와 동일)
# =======================================================
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return dict(accuracy=accuracy_score(predictions, labels))

args = TrainingArguments(
    f"custom-vit-finetune",
    save_strategy="epoch",
    eval_strategy="epoch", # 'evaluation_strategy' 대신 'eval_strategy' 사용
    learning_rate=2e-5,
    per_device_train_batch_size=10,
    per_device_eval_batch_size=4,
    num_train_epochs=5, # CIFAR-10보다 데이터가 적으므로 에포크를 늘릴 수 있음
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir='logs_custom',
    remove_unused_columns=False,
    report_to=["tensorboard"], # W&B 오류 방지
)

In [17]:
trainer = Trainer(
    model,
    args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    tokenizer=processor,
)

print("\n--- 커스텀 데이터셋 파인 튜닝 시작 ---")
trainer.train()

/tmp/ipython-input-3793138566.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



--- 커스텀 데이터셋 파인 튜닝 시작 ---


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.592092,0.965517
2,No log,0.420261,0.965517
3,No log,0.342000,0.965517
4,No log,0.301854,0.965517
5,No log,0.291767,0.965517


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=135, training_loss=0.4082018534342448, metrics={'train_runtime': 3090.5158, 'train_samples_per_second': 0.422, 'train_steps_per_second': 0.044, 'total_flos': 1.0112795281677312e+17, 'train_loss': 0.4082018534342448, 'epoch': 5.0})

In [19]:
print("\n--- 최종 모델 성능 평가 시작 ---")

# 1. 검증 데이터셋 (val_ds)에 대한 평가
# trainer.evaluate()를 호출하여 compute_metrics에서 정의한 정확도를 계산합니다.
results = trainer.evaluate(eval_dataset=val_ds)

# 2. 결과 출력
# results 딕셔너리에는 eval_loss, eval_accuracy 등의 정보가 포함되어 있습니다.
print(results)

# 3. 최종 정확도 명시적 출력
final_accuracy = results['eval_accuracy']
print(f"\n✅ 최종 모델의 검증 정확도 (Accuracy): {final_accuracy:.4f}")


--- 최종 모델 성능 평가 시작 ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.5920922756195068, 'eval_accuracy': 0.9655172413793104, 'eval_runtime': 32.1036, 'eval_samples_per_second': 0.903, 'eval_steps_per_second': 0.249, 'epoch': 5.0}

✅ 최종 모델의 검증 정확도 (Accuracy): 0.9655


In [18]:
# 학습 완료 후 모델 저장 (선택 사항)
output_dir = "final_custom_vit_model"
trainer.model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)
print(f"\n✅ 최종 모델이 {output_dir}에 저장되었습니다.")


✅ 최종 모델이 final_custom_vit_model에 저장되었습니다.
